# Cross-model tournament — 260723

Runs the shortlisted `260723` models against each other in inference and writes one behavioural-log JSON per matchup (the format `octagon_analysis` consumes).

**Where the code lives.** This notebook is config + visualisation only. Everything that runs or checks a matchup is in the two shared modules:
- `tournament/tournament_func.py` — assembling a matchup run dir, `run_matchup`, `run_tournament`.
- `tournament/tournament_checks.py` — `preflight`, `smoke_test`, `check_batch`, `rerun_matchups`.

Analysis of the resulting logs is `octagon_analysis/analysis/tournament_analysis.py` + `demos/tournament/tournament_analysis_260723.ipynb`.

**What is different about this batch.** The `260723` models were trained on `build_260723_01_social`, with the no-movement penalty **break removed** — every step is penalised, resting included. The tournament build (`build_260724_01_tournament`) has the break absent too, so for this batch the tournament reward structure **matches** what the models were trained under. That was not true of `260715` (trained with the break present), where the reward analysis had to be held back. So the reward ranking is worth reading here, not just the score ranking.

**Before running:**
1. Activate the conda env that has `mlagents` (the kernel for this notebook must be that env).
2. Sanity-check CUDA (`nvidia-smi`, `torch.cuda.is_available()`) — see the driver notes in `CLAUDE.md`.
3. The build below must be the tournament build with `ScriptedTrialSequence.cs` + the timing keys; an older build silently ignores `--trial_seq` and generates random trials.

Actions are **sampled** (not greedy), matching training behaviour → matchups are not bit-reproducible; run enough episodes to average over sampling noise. Trial content/timing is pinned by the predetermined `trial_seq`.

In [1]:
import importlib
import sys
from pathlib import Path

# Repo root on sys.path so `tournament.*` and the `trainer_and_simulator_functions`
# they import both resolve.
REPO_ROOT = Path("~/repos/agent_training").expanduser()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Force a reload of all three modules. Python caches imports, so editing tournament_func.py
# has NO effect on an already-running kernel. That is how the 260730 batch was lost: the fixed
# file was on disk, but the kernel kept the pre-fix functions in memory, wrote configs with no
# environment_parameters, and froze P2 in all 64 matchups a second time. Reload here, and if in
# any doubt restart the kernel outright.
import trainer_and_simulator_functions as _tsf
importlib.reload(_tsf)
import tournament.tournament_func as tf
importlib.reload(tf)
import tournament.tournament_checks as tc
importlib.reload(tc)

from tournament.tournament_func import run_tournament

## Config

Everything batch-specific lives in this cell. `MODELS_ROOT` is the parent folder; entrants are its immediate subfolders (each with `OctagonAgentSocial/checkpoint.pt`). All entrants must live under one `MODELS_ROOT` — to mix `260715` and `260723`, gather the chosen run dirs under a single folder first.

In [ ]:
OCTAGON_DIR = Path("~/Unity/Octagon").expanduser()

# Tournament build (headless Linux). Penalty break ABSENT, which matches how the 260723
# models were trained — so tournament reward is comparable to training reward for this batch.
UNITY_ENV_PATH = Path("~/Unity/Octagon/builds/260909/build_260909_01_tournament_nosteppenalty.x86_64").expanduser()

# Parent folder holding the entrant run dirs (each an immediate subfolder).
MODELS_ROOT = Path("~/Unity/Octagon/results/2607/260723").expanduser()

# Hand-picked entrants, from the Stage-4 shortlist in
# octagon_analysis/demos/strategy_characterisation/competitive/model_characterisation_demo260723.ipynb.
# Listed central -> intermediate -> peripheral so the analysis matrices read in strategy order.
# Set to None to auto-use EVERY model under MODELS_ROOT (careful: N models => C(N,2)+N matchups).
RUN_IDS = [
    "260723_0034",  # central
    "260723_0039",  # central
    "260723_0044",  # intermediate (the only one that earned a place in the gap)
    "260723_0035",  # peripheral, top of the shortlist
    "260723_0001",  # peripheral
    "260723_0033",  # peripheral
    "260723_0051",  # peripheral
    "260723_0023",  # peripheral
]

# Predetermined trial sequence (pins trial content + timing across all matchups).
# Keep it LONGER than EPISODES — the build stops when the sequence is exhausted.
TRIAL_SEQ = Path("~/repos/agent_training/trial_sequences/trials_100000_seed17.json").expanduser()

# Where the per-matchup JSON logs land: one folder per batch, so a round-robin is self-contained.
OUT_ROOT = Path("~/Unity/Octagon/simulations/tournaments/260723").expanduser()

# MLAgents episodes per matchup. Conversion (measured on 260715 inference): trials ~= episodes - 1.
# So for a target of T trials/matchup, set EPISODES ~= T + 1. Same value as 260715, so the two
# batches carry equal weight per matchup.
EPISODES = 350   # ~350 trials/matchup

# Run each distinct pair in BOTH seat orders (mirror matchups). P1/P2 is symmetric so this is
# off in the function by default; ON here to double data (2 x ~350 = ~700 trials/pair) and to
# confirm the seat has no effect by comparing a pair against its mirror. On 260715 the mean
# mirror gap came out at 0.018, so the assumption held.
INCLUDE_MIRRORS = True

# Include models playing against themselves — one per entrant, giving a ~50% baseline check.
INCLUDE_SELF_MATCHUPS = False

In [ ]:
# Fail fast if a path is missing, an entrant has no checkpoint, or — the one that matters —
# the generated config has no environment_parameters/step_penalty block. Without it the build
# throws on every action, CompetitiveAgent2 never gets stepped, and every matchup is a
# walkover that still writes a complete, healthy-looking JSON. The check runs from notebook
# source so it also catches a kernel holding pre-fix code, which an assertion inside
# tournament_func.py could not.
n_matchups = tc.preflight(
    models_root=MODELS_ROOT,
    unity_env_path=UNITY_ENV_PATH,
    trial_seq=TRIAL_SEQ,
    run_ids=RUN_IDS,                        # None => every model under MODELS_ROOT
    include_mirrors=INCLUDE_MIRRORS,
    include_self_matchups=INCLUDE_SELF_MATCHUPS,
)

## (Optional) Single-matchup smoke test

Run one matchup with a few episodes first, to confirm the build + checkpoints load and both agents actually play, before committing to the full round-robin (~64 matchups at ~35 min each).

In [ ]:
# Both agents must MOVE, the wall triggers must be split, and Unity must not be throwing on
# step_penalty. Raises if not. A frozen opponent still produces a complete, full-length JSON
# and a clean exit — that is exactly how two 64-matchup batches were lost.
health = tc.smoke_test(
    model_a_run_dir=MODELS_ROOT / RUN_IDS[0],
    model_b_run_dir=MODELS_ROOT / RUN_IDS[1],
    octagon_dir=OCTAGON_DIR,
    unity_env_path=UNITY_ENV_PATH,
    out_root=OUT_ROOT,
    trial_seq=TRIAL_SEQ,
    episodes=8,
)

## Full round-robin

Every unordered pair, **including self-matchups**, and (with `INCLUDE_MIRRORS`) each pair in both seat orders. One JSON per matchup under `OUT_ROOT/<A>__vs__<B>/`. Failures are caught per-matchup (that entry is `None`) so one bad run doesn't kill the batch.

**Self-matchups vs mirrors — to be clear:**
- **Self-matchups (a model vs a copy of itself) ARE run** — one per entrant, giving a ~50% baseline / sanity check.
- **Mirrors (the same pairing with the P1/P2 seats swapped)** are off in the function by default. Which seat a model takes should be irrelevant: agents aren't repositioned at trial start, so each agent's position is emergent from its own behaviour, and the only P1/P2 difference is administrative (arena setup / logging). They are ON here to double the data per pair and to re-confirm empirically that the seat has no effect.

In [ ]:
logs = run_tournament(
    models_root=MODELS_ROOT,
    octagon_dir=OCTAGON_DIR,
    unity_env_path=UNITY_ENV_PATH,
    out_root=OUT_ROOT,
    episodes=EPISODES,
    run_ids=RUN_IDS,            # None => all models under MODELS_ROOT
    include_self_matchups=INCLUDE_SELF_MATCHUPS,
    include_mirrors=INCLUDE_MIRRORS,
    trial_seq=TRIAL_SEQ,
    base_port=9100,             # ports = 9100 + 20*i -> 9100..10360. Training uses
                                # 5005+20*i per model, so a session tops ~6400 (70 models)
                                # / ~9000 (200 models); 9100 clears sessions up to ~206
                                # models. Also above tensorboard (6006) / mDNS (5353).
)

In [ ]:
# Which matchups ran, and — more importantly — which are VALID. A matchup can exit cleanly
# with a complete 350-trial JSON and still be void: before the 260729 fix a missing
# environment_parameters block froze P2 at its spawn in all 64 matchups and handed P1 every
# trial, and nothing in the exit status showed it. Only suspect matchups are printed.
suspect = tc.check_batch(
    out_root=OUT_ROOT,
    tournament_ids=logs.keys(),
    expected_trials=EPISODES,
    octagon_dir=OCTAGON_DIR,
    logs=logs,                              # so matchups that raised are reported too
)

## Rerun specific matchups

For matchups that failed or came out short above, without re-running the whole batch. List the ordered `(P1, P2)` pairs in `RERUN` — the same ids that make up the matchup dir name `<P1>__vs__<P2>`.

Everything else (build, checkpoints, trial sequence, sampled actions) is identical to the round-robin, so the reruns are drop-in replacements for the originals.

**On stalled matchups.** In `260715`, the `0002` self-matchup did not crash — it stalled: `No episode was completed since last summary` in its stdout, **68** `trial start` events in ~7200 s against ~350 in ~2200 s for a healthy matchup. Both copies wandered without triggering a wall. Actions are sampled, so a **different `seed` re-rolls the trajectories** and is the thing most likely to change the outcome; raising `timeout_s` alone mostly buys a longer stall. Check the trial count in the health line, not just "no exception".

In [ ]:
# Ordered (P1, P2) run_id pairs to rerun. Same id twice => self-matchup.
RERUN = [
    # ("260723_0034", "260723_0034"),
]

rerun_logs = tc.rerun_matchups(
    pairs=RERUN,
    models_root=MODELS_ROOT,
    octagon_dir=OCTAGON_DIR,
    unity_env_path=UNITY_ENV_PATH,
    out_root=OUT_ROOT,          # same dir as the round-robin, so reruns land beside siblings
    episodes=EPISODES,          # trials ~= episodes - 1
    trial_seq=TRIAL_SEQ,
    seed=17,                    # change to re-roll the sampled trajectories
    timeout_s=5000,             # wall-clock cap per matchup (healthy 350-ep run: ~2200 s)
    base_port=10500,            # clear of the round-robin's 9100..10360
    archive_existing=True,      # move the old matchup dir to OUT_ROOT/../superseded/ first
)